# 00 — SDS EXAM PLAYBOOK (read the morning of the exam)
Jojie + Respondus · allowed: Jojie, GitHub, Copilot · **built-in Apache Spark only (no GraphFrames, no Spark packages)** · **no LLMs.**

## The exam is two skills
1. **~40% — critique/fix an LLM's streaming/sketching answer** (P1a + all P2). Notebook 01. Know each algorithm's correct params + the one gotcha; run a verifier to *prove* the bug.
2. **~60% — implement graph/similarity algorithms in pure PySpark** (P1b, P3, P4). Notebooks 02–04.

## Phrase → cell lookup
| Exam phrase | Notebook · cell |
|---|---|
| "critique / fix" + LSH | 01 style + 02 · 1a S-curve tables |
| "critique / fix" + sample/Bloom/reservoir/HLL/IQR | 01 · matching verifier |
| find most similar document | 02 · `find_similar` |
| compute PageRank, top-25 | 03 · PageRank |
| hubbiness & authority (HITS) | 03 · HITS |
| advice to CEO from results | 03 · 3c formula |
| Girvan-Newman partition | 04 · subset + GN |
| SimRank partition | 04 · subset + SimRank |
| clustering coefficient | 04 · full-graph Spark |

## The three reflexes that win points
1. **Subset-with-justification.** GN is O(V·E)/removal, SimRank O(n²) — infeasible at ~600K nodes. Induce the **top-N-by-degree subgraph in Spark**, state the complexity reason in a comment, run the exact algorithm on the core, and say what the subset represents. (PageRank/HITS/clustering-coeff DO scale → run on the full graph.)
2. **Small corpus ⇒ exact beats approximate — and say so.** `find_similar` on ~100 docs: exact cosine over TF-IDF, note you'd switch to LSH at scale.
3. **Honest "worse" result still scores.** If SimRank fragments more than GN, or a method underperforms, state it plainly with the reason.

## Data-inspection first (every Spark problem)
Print schema + `groupBy("type").count()` before modeling. The clickstream is a **TSV** — reading it as CSV dumps everything into one column; re-read with `sep="\t"` and name columns `src,dst,type,count`, filter `type='link'`.

## Non-negotiable correctness details (graders look for these)
- PageRank: teleport (d=0.85) **and** redistribute **dangling mass** (else ranks leak; sum<1). ~15 iters.
- HITS: **L2-normalize every iteration**, no teleport; **persist+checkpoint+count each iteration** or RDD lineage explodes and it hangs.
- GN: K at **peak modularity Q**; SimRank: θ at the **K–θ plateau**, c=0.8.
- Clustering coefficient: symmetrize, self-join for 2-paths, join again for closed; near-zero CC is expected for a navigational graph.
- **Numbers in your prose must equal what the code printed** (the past exam's write-up drifted from its output — easy points lost).

## Streaming critique cheat row (full table in nb 01)
Bloom no-false-negatives · reservoir all-time-uniform-not-window · HLL SE=1.04/√m & needs α_m bias correction · Count-Min never undercounts · hash%1≡0 samples 100% · Bloom m=⌈−n·lnε/(ln2)²⌉ k=round((m/n)ln2) · running-IQR insert is O(W) not O(log W).

## Time budget (adapt to the actual point split; past exam was 4×25)
| Block | Time |
|---|---|
| Clone repo on Jojie, start Spark, env check | 15 min |
| Critique/streaming problems (fast if you know the gotchas) | ~45 min |
| PageRank + HITS (+ advice) | ~60 min |
| GN + SimRank + clustering coeff (subset first!) | ~70 min |
| Final pass: every cell executed? prose numbers = outputs? | 15 min |

**If a Spark job hangs >5 min:** you likely forgot to materialize an iterative RDD (persist+checkpoint+count) or you're `.collect()`-ing something huge — shrink N, subset, or cache, state the simplification, move on.

## Copilot (your only allowed AI)
Write a complete specific comment first (`# PageRank power iteration in Spark: contrib = rank/outdeg along edges, sum at dst, teleport 0.15/N + dangling`), then accept. Keep notebooks 01–04 open in tabs so it autocompletes in this style. Run every cell — it invents column names.